In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import yaml
import glob
import re
import os
import time

load_dotenv()
client = OpenAI()

def get_latest_prompt_file():
    """가장 높은 버전의 prompt yaml 파일을 찾아 반환"""
    prompt_files = glob.glob("./prompt/prompt_ver*.yaml")
    
    if not prompt_files:
        print("프롬프트 파일을 찾을 수 없습니다.")
        return None
    
    # 버전 번호 추출하여 정렬
    def extract_version(filename):
        match = re.search(r'prompt_ver(\d+)\.yaml', filename)
        return int(match.group(1)) if match else 0
    
    latest_file = max(prompt_files, key=extract_version)
    print(f"사용 중인 프롬프트 파일: {latest_file}")
    return latest_file

def load_prompt_config(prompt_file_path):
    """YAML 프롬프트 설정 파일을 로드"""
    with open(prompt_file_path, 'r', encoding='utf-8') as file:
        return yaml.safe_load(file)
        
def read_txt_file(file_path):
    """txt 파일을 읽어서 내용을 반환합니다."""
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read().strip()

def save_processed_text(original_file_path, processed_text):
    """처리된 텍스트를 ./processed_output 디렉토리에 저장"""
    # processed_output 디렉토리 생성
    output_dir = "./processed_output"
    os.makedirs(output_dir, exist_ok=True)
    
    # 원본 파일명에서 확장자 제거하고 _processed 추가
    base_name = os.path.basename(original_file_path)
    name_without_ext = os.path.splitext(base_name)[0]
    output_filename = f"{name_without_ext}_processed.txt"
    output_path = os.path.join(output_dir, output_filename)
    
    with open(output_path, 'w', encoding='utf-8') as file:
        file.write(processed_text)
    return output_path

def get_all_txt_files():
    """./ocr_output 폴더의 모든 .txt 파일 경로를 반환"""
    ocr_output_dir = "./ocr_output"
    if not os.path.exists(ocr_output_dir):
        print(f"입력 폴더가 존재하지 않습니다: {ocr_output_dir}")
        return []
    
    txt_files = glob.glob(os.path.join(ocr_output_dir, "*.txt"))
    return txt_files

def generate_response_from_txt(txt_file_path, prompt_config):
    """txt 파일의 내용을 프롬프트와 함께 ChatGPT API 호출"""
    
    # txt 파일 내용 읽기
    txt_content = read_txt_file(txt_file_path)
    if txt_content is None:
        return None
    
    # 프롬프트 구성
    user_prompt = prompt_config['user_prompt_template'].format(ocr_text=txt_content)
    
    if 'additional_instructions' in prompt_config:
        user_prompt += f"\n\n{prompt_config['additional_instructions']}"

    try:
        completion = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system", 
                    "content": prompt_config['system_prompt']
                },
                {
                    "role": "user", 
                    "content": user_prompt
                },
            ],
            # max_tokens=4000,
            temperature=0.1
        )
        
        processed_text = completion.choices[0].message.content
        
        # 처리된 텍스트를 파일로 저장
        save_processed_text(txt_file_path, processed_text)
        
        return processed_text
    
    except Exception as e:
        print(f"API 호출 오류: {e}")
        return None

def process_all_files():
    """./ocr_output의 모든 .txt 파일을 처리"""
    txt_files = get_all_txt_files()
    
    if not txt_files:
        print("처리할 .txt 파일이 없습니다.")
        return

    prompt_file = get_latest_prompt_file()
    if not prompt_file:
        return
    
    prompt_config = load_prompt_config(prompt_file)
    if not prompt_config:
        return
    
    print(f"총 {len(txt_files)}개 파일을 처리합니다.")
    print("-" * 50)
    
    successful_count = 0
    failed_count = 0
    
    for i, txt_file_path in enumerate(txt_files, 1):
        filename = os.path.basename(txt_file_path)
        print(f"[{i}/{len(txt_files)}] {filename} ", end="", flush=True)
        
        start_time = time.time()
        
        try:
            result = generate_response_from_txt(txt_file_path, prompt_config)
            
            if result:
                print(" ✓ 완료")
                successful_count += 1
            else:
                print(" ✗ 실패")
                failed_count += 1
                
        except Exception as e:
            print(f" ✗ 오류: {e}")
            failed_count += 1
        
        # API 호출 제한을 위한 대기
        if i < len(txt_files):
            time.sleep(1)
    
    print("\n" + "="*50)
    print("처리 완료!")
    print(f"성공: {successful_count}개")
    print(f"실패: {failed_count}개")
    print(f"전체: {len(txt_files)}개")

# 사용 예시
if __name__ == "__main__":
    process_all_files()